In [ ]:
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.agent.conversation_manager import (
    SlidingWindowConversationManager,
    SummarizingConversationManager,
)
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands_tools import think
from IPython.display import Markdown, display
from dotenv import load_dotenv
import os

load_dotenv()
YOUR_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-5"

brave_api_key = os.getenv("BRAVE_API_KEY")

In [ ]:
# Python MCP server
calculator_server_path = "calculator_mcp.py"
calculator_server = {
    "type": "stdio",
    "command": "python",
    "args": [calculator_server_path]
}
print(f"Calculator server configured: {'Yes' if calculator_server else 'No'}")


# Brave MCP server written in TypeScript

brave_api_key = os.environ.get("BRAVE_API_KEY", "")
print(f"Brave API key available: {'Yes' if brave_api_key else 'No'}")
brave_search_server = {
    "type": "stdio",
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-brave-search"],
    "env": {
        "BRAVE_API_KEY": brave_api_key, 
        "PATH": f"{os.path.dirname('npx')}:" + os.environ.get("PATH", "")
    }
}
print(f"Brave search server configured: {'Yes' if brave_search_server else 'No'}")

Calculator server configured: Yes
Brave API key available: Yes
Brave search server configured: Yes


In [ ]:
system_prompt = """
You are a helpful assistant with access to:
1. Web search (brave_web_search, brave_local_search)
2. Mathematical calculator (calculator)
3. A tool to think and reason (think)

Always use the most appropriate tool for each task.
"""

# Bedrock model (replace with the model/inference profile ID for your region)
# context_window_limit is what the conversation managers below measure against:
# with proactive_compression they reduce the history before any model call that
# is projected to use more than 70% of it.
model = BedrockModel(
    model_id=YOUR_BEDROCK_MODEL_ID,
    region_name=os.getenv("AWS_REGION", "us-west-2"),
    max_tokens=4096,
    context_window_limit=200_000,
)

# MCP servers
import sys
from pathlib import Path

brave_search_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-brave-search"],
        env={**os.environ, "BRAVE_API_KEY": os.getenv("BRAVE_API_KEY", "")},
    )
))

# Run the local calculator server with this kernel's own interpreter.
# `uvx` lives in ~/.local/bin, which is not on the Jupyter kernel's PATH,
# so `command="uvx"` raises FileNotFoundError when the client starts.
calculator_path = str(Path(calculator_server_path).resolve())

calculator_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command=sys.executable,
        args=[calculator_path],
        env={**os.environ},
    )
))

In [ ]:
def call_cagent(prompt, conversation_manager=None):
    # MCP clients must be open while the agent runs
    with brave_search_client, calculator_client:
        tools = (
            [think]
            + brave_search_client.list_tools_sync()
            + calculator_client.list_tools_sync()
        )

        # Keeps the history to the last 20 messages, and trims it again before
        # any model call projected to pass 70% of the context window. Search
        # results are what fill a window here, so the trim targets the oldest
        # tool results first and never orphans a toolUse from its toolResult.
        # A manager holds per-run state, so each agent gets its own.
        agent = Agent(
            name="Multi-Tool Agent",
            model=model,
            system_prompt=system_prompt,
            tools=tools,
            conversation_manager=conversation_manager
            or SlidingWindowConversationManager(
                window_size=20, proactive_compression=True
            ),
            callback_handler=None,  # don't also stream the answer to stdout
        )

        # Tools the agent can call
        display(Markdown(
            f"**{len(agent.tool_names)} tools available:**\n\n"
            + "\n".join(f"- `{name}`" for name in agent.tool_names)
        ))

        response = agent(prompt)

    # Tools the agent actually called (absent from the dict means never called)
    used = response.metrics.tool_metrics
    display(Markdown(
        f"**{len(used)} tools used:**\n\n"
        + ("\n".join(
            f"- `{name}` — {m.call_count} call(s), {m.success_count} ok, "
            f"{m.error_count} error(s), {m.total_time:.2f}s"
            for name, m in used.items()
        ) or "_none_")
    ))

    # What the history cost, and what the manager dropped to keep it in bounds
    usage = response.metrics.accumulated_usage
    display(Markdown(
        f"**Context:** {len(agent.messages)} messages kept, "
        f"{agent.conversation_manager.removed_message_count} removed "
        f"— {usage['inputTokens']:,} input / {usage['outputTokens']:,} "
        f"output tokens"
    ))

    display(Markdown(str(response)))

In [ ]:
prompt = "Search the web for the population of Uruguay and multiply it by 1.5"
call_cagent(prompt)

**4 tools available:**

- `think`
- `brave_web_search`
- `brave_local_search`
- `calculator`

**2 tools used:**

- `brave_web_search` — 1 call(s), 1 ok, 0 error(s), 0.61s
- `calculator` — 1 call(s), 1 ok, 0 error(s), 0.00s

**Context:** 6 messages kept, 0 removed — 9,865 input / 423 output tokens

## Result

**Population of Uruguay (2024):** approximately **3,423,316** people (per Macrotrends/UN data; estimates range from ~3.39M to ~3.49M depending on the source).

**Multiplied by 1.5:**
3,423,316 × 1.5 = **5,134,974**

Note: Different sources give slightly varying figures (e.g., Worldometer ~3.39M, countryeconomy.com ~3.49M for 2024), so this result would shift slightly depending on which population estimate you use.


In [ ]:
prompt = """Search for the current population of Tokyo, Japan. 
Then calculate how many people that would be per square kilometer, 
given that Tokyo's area is approximately 2,194 square kilometers."""
call_cagent(prompt)

**4 tools available:**

- `think`
- `brave_web_search`
- `brave_local_search`
- `calculator`

**2 tools used:**

- `brave_web_search` — 1 call(s), 1 ok, 0 error(s), 0.67s
- `calculator` — 1 call(s), 1 ok, 0 error(s), 0.00s

**Context:** 6 messages kept, 0 removed — 9,897 input / 762 output tokens

## Tokyo Population & Density Summary

**Current Population of Tokyo (Tokyo Metropolis):**
Based on recent estimates, Tokyo's population is approximately **14,264,798 people** (2026 estimate). This refers to the Tokyo Metropolis (the broader prefecture-level administrative area), not just the greater metropolitan area (which includes surrounding prefectures and totals around 37 million people — one of the largest urban agglomerations in the world).

**Population Density Calculation:**

Given:
- Population: 14,264,798
- Area: 2,194 km²

$$
\text{Density} = \frac{14,264,798}{2,194} \approx 6,502 \text{ people per km}^2
$$

**Result: Tokyo has a population density of approximately 6,502 people per square kilometer.**

### Context
This is an extremely high density — for comparison:
- New York City: ~11,000 people/km²
- Los Angeles: ~3,200 people/km²
- London: ~5,700 people/km²

Tokyo's density reflects its status as one of the most populous urban centers on Earth, though it's worth noting that density varies dramatically within Tokyo itself — the 23 central wards are far denser (often 15,000+ people/km²) than the outer, more suburban/mountainous parts of the Tokyo Metropolis that bring the overall average down.


## Keeping a long chat in context

`SlidingWindowConversationManager` above drops the oldest messages, which is
cheap but loses whatever was in them. Over a multi-turn chat,
`SummarizingConversationManager` is usually the better trade: when the history
has to shrink, it asks the model to summarize the oldest messages and replaces
them with that summary, so earlier findings survive in compressed form.

Both take `proactive_compression=True`, which reduces the history *before* a
model call that is projected to cross 70% of `context_window_limit`, rather
than waiting for the provider to reject an oversized request.

In [ ]:
# A deliberately small window, so compression triggers within a four-turn chat
# instead of after hundreds of turns. Production code sets the model's real
# limit (200k above) and lets the threshold do the work.
chat_model = BedrockModel(
    model_id=YOUR_BEDROCK_MODEL_ID,
    region_name=os.getenv("AWS_REGION", "us-west-2"),
    max_tokens=4096,
    context_window_limit=8_000,
)

summarizer = SummarizingConversationManager(
    summary_ratio=0.5,            # summarize the oldest half of the history
    preserve_recent_messages=4,   # keep the last two exchanges verbatim
    proactive_compression=True,   # compress before a call crosses the threshold
)

questions = [
    "Search for the current population of Uruguay.",
    "Now search for the population of Paraguay.",
    "What is the combined population of the two countries?",
    "Which one is larger, and by what percentage?",
]

# One agent for the whole chat, so the history accumulates across turns
with brave_search_client, calculator_client:
    tools = (
        [think]
        + brave_search_client.list_tools_sync()
        + calculator_client.list_tools_sync()
    )

    chat_agent = Agent(
        name="Research Chat",
        model=chat_model,
        system_prompt=system_prompt,
        tools=tools,
        conversation_manager=summarizer,
        callback_handler=None,
    )

    for question in questions:
        response = chat_agent(question)
        usage = response.metrics.accumulated_usage
        display(Markdown(
            f"**Q:** {question}\n\n{response}\n\n"
            f"_{len(chat_agent.messages)} messages in history, "
            f"{summarizer.removed_message_count} replaced by a summary, "
            f"{usage['inputTokens']:,} input tokens so far_"
        ))

# Once the manager has compressed, the summary it wrote is the first message
first_text = chat_agent.messages[0]["content"][0].get("text", "")
display(Markdown(
    f"**Running summary**\n\n{first_text}"
    if first_text.startswith("## Conversation Summary")
    else "_No compression needed: the chat stayed under the threshold._"
))

**Q:** Search for the current population of Uruguay.

## Current Population of Uruguay

Based on recent data, the population of Uruguay is approximately **3.4 million people**. Here's a breakdown from various sources:

| Source | Population | Year |
|--------|-----------|------|
| Macrotrends | 3,423,316 | 2024 |
| World Bank | 3,386,588 | 2024 |
| PopulationPyramid.net | 3,386,587 | 2024 |
| StatisticsTimes | ~3.39 million | 2024 |
| CountryEconomy.com | 3,486,000 | 2025 |

**Key facts:**
- Uruguay is the **10th-most populous country in South America**
- It ranks around **134th in the world** by population
- The median age is approximately **37.2 years**, reflecting a relatively mature population
- Growth has been nearly flat in recent years — Macrotrends notes only a **0.01% increase** from 2023 to 2024, and some sources even show a slight **decrease** heading into 2025

**Bottom line:** Uruguay's population is right around **3.4 million people**, with growth essentially stagnant in recent years.


_4 messages in history, 0 replaced by a summary, 5,949 input tokens so far_

**Q:** Now search for the population of Paraguay.

## Current Population of Paraguay

Based on recent data, Paraguay's population is approximately **6.9 million people**. Here's a breakdown from various sources:

| Source | Population | Year |
|--------|-----------|------|
| Macrotrends | 6,947,270 | 2024 |
| World Bank | 6,929,153 | 2024 |
| StatisticsTimes | ~6.93 million | 2024 |
| Statista | ~6.93 million | 2024 |
| CountryEconomy.com | 6,373,000 | 2024 |
| database.earth (projection) | 7,095,279 | 2026 |

**Key facts:**
- Paraguay is the **9th-most populous country in South America**
- It ranks around **108th in the world** by population
- Growth is still fairly steady — roughly **1.25% increase** from 2023 to 2024, with population split nearly evenly between men (~3.46 million) and women (~3.47 million)
- Over the past decade, growth has been driven mostly by natural increase (births over deaths), with a net loss from migration

**Comparison with Uruguay:** Paraguay's population (~6.9 million) is roughly **double** that of Uruguay (~3.4 million), despite Uruguay having a slightly higher global population rank per capita in terms of density of statistics coverage — Paraguay is simply a larger country by population while Uruguay's growth has nearly stalled in comparison.


_8 messages in history, 0 replaced by a summary, 15,621 input tokens so far_

**Q:** What is the combined population of the two countries?

## Combined Population of Uruguay and Paraguay

Using matched data sources for consistency:

**Using Macrotrends figures (2024):**
- Uruguay: 3,423,316
- Paraguay: 6,947,270
- **Combined: 10,370,586**

**Using World Bank figures (2024):**
- Uruguay: 3,386,588
- Paraguay: 6,929,153
- **Combined: 10,315,741**

### Summary
The combined population of Uruguay and Paraguay is approximately **10.3 to 10.4 million people**, depending on the data source used. This places the two countries together in the same population range as a mid-sized country — for context, roughly comparable to the population of Sweden or Portugal.

Paraguay contributes about two-thirds of this combined total, while Uruguay accounts for roughly one-third, reflecting Paraguay's considerably larger population base.


_9 messages in history, 4 replaced by a summary, 25,125 input tokens so far_

**Q:** Which one is larger, and by what percentage?

## Paraguay is Larger

**Paraguay has a significantly larger population than Uruguay** — roughly double in size.

### Percentage Difference

**Using Macrotrends data (2024):**
- Paraguay: 6,947,270
- Uruguay: 3,423,316
- Difference: 3,523,954
- **Paraguay is ~102.9% larger than Uruguay** (i.e., more than double)

**Using World Bank data (2024):**
- Paraguay: 6,929,153
- Uruguay: 3,386,588
- Difference: 3,542,565
- **Paraguay is ~104.6% larger than Uruguay**

### Summary
Regardless of the data source, **Paraguay's population is approximately 103–105% larger than Uruguay's** — meaning Paraguay has roughly **2.0 to 2.05 times** the population of Uruguay. This aligns with their respective rankings in South America, where Paraguay (9th) outranks Uruguay (10th) in total population.


_8 messages in history, 11 replaced by a summary, 40,205 input tokens so far_

**Running summary**

## Conversation Summary
* Topic 1: User requested a search for the current population of Uruguay.
* Topic 2: Compilation and comparison of population figures from multiple demographic data sources for Uruguay.
* Topic 3: User requested a search for the current population of Paraguay.
* Topic 4: Compilation and comparison of population figures from multiple demographic data sources for Paraguay.
* Topic 5: User requested the combined population of Uruguay and Paraguay.
* Topic 6: Calculation of combined totals using matched data sources (Macrotrends and World Bank).

## Tools Executed
* **brave_web_search** (query: "current population of Uruguay 2024"): Returned multiple sources including Worldometer, Wikipedia, Macrotrends, StatisticsTimes.com, PopulationPyramid.net, database.earth, U.S. Census Bureau Population Clock, Trading Economics, World Bank Data, and CountryEconomy.com.
* **brave_web_search** (query: "current population of Paraguay 2024"): Returned multiple sources including Macrotrends, Worldometer, CountryEconomy.com, PassportRanker, StatisticsTimes.com, Statista, database.earth, GeoRank, and Trading Economics.
* **calculator** (3,423,316 + 6,947,270): Result = 10,370,586 (using Macrotrends figures for both countries)
* **calculator** (3,386,588 + 6,929,153): Result = 10,315,741 (using World Bank figures for both countries)

## Technical/Data Information Shared
**Uruguay:**
* Macrotrends: 3,423,316 (2024), a 0.01% increase from 2023
* World Bank: 3,386,588 (2024)
* PopulationPyramid.net: 3,386,587 (2024), median age 37.2 years
* StatisticsTimes: ~3.39 million (2024)
* CountryEconomy.com: 3,486,000 (2025)
* Ranked 10th-most populous country in South America, ~134th globally

**Paraguay:**
* Macrotrends: 6,947,270 (2024), a 1.25% increase from 2023
* World Bank: 6,929,153 (2024)
* StatisticsTimes: ~6.93 million (2024)
* Statista: ~6.93 million (2024), split ~3.46 million men, ~3.47 million women
* CountryEconomy.com: 6,373,000 (2024)
* database.earth projection: 7,095,279 (2026)
* Ranked 9th-most populous country in South America, ~108th globally

**Combined Population:**
* Using Macrotrends data: approximately 10,370,586
* Using World Bank data: approximately 10,315,741

## Key Insights Gained
* Uruguay's population (~3.4 million) is roughly half that of Paraguay's (~6.9 million).
* Both countries show relatively modest population growth, with Paraguay growing faster (1.25%) than Uruguay (0.01%) year-over-year.
* The combined population of Uruguay and Paraguay is approximately 10.3–10.4 million people, depending on the data source used.
* Paraguay ranks higher in South America (9th) and globally (108th) in population compared to Uruguay (10th in South America, ~134th globally).
* Data consistency varies by source, with Macrotrends and World Bank figures showing slight differences for both countries.